# Normalización del dataset de NASA

### 1. Importar librerias necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import os

from pathlib import Path
from difflib import get_close_matches

### 2. Carga del dataset a normalizar

In [ ]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\01 - Originales\05 - Nasa\05 - NASA.parquet'

# Cargar el archivo CSV directamente
df = pd.read_parquet(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

# Mostrar el número de municipios únicos en la columna 'municipio'
print(f"Municipios únicos en el dataset: {df['municipio'].nunique()}")

Filas cargadas: 755178
Columnas disponibles: ['fecha', 'municipality_code', 'lat', 'lon', 'geometry', 'ALLSKY_SFC_SW_DWN', 'T2MWET', 'PS', 'QV2M', 'municipio']
Tamaño del dataset: 755178 filas x 10 columnas


,fecha,municipality_code,lat,lon,geometry,ALLSKY_SFC_SW_DWN,T2MWET,PS,QV2M,municipio
0,2000-01-01,15001,43.208955,-8.294335,MULTIPOLYGON (((-8.270063663999963 43.28377445...,3.75,1.93,97.56,3.98,abegondo
1,2000-01-01,15002,42.890631,-8.645921,MULTIPOLYGON (((-8.630711194999947 42.94959652...,7.97,5.15,100.18,4.94,ames
2,2000-01-01,15003,43.222655,-8.012332,MULTIPOLYGON (((-7.921458001999952 43.26941245...,3.75,1.93,97.56,3.98,aranga
3,2000-01-01,15004,43.437931,-8.257343,MULTIPOLYGON (((-8.279739824999979 43.45941986...,3.75,5.65,100.50,5.02,ares
4,2000-01-01,15006,42.926506,-8.182868,MULTIPOLYGON (((-8.204219274999957 42.98504724...,7.97,1.93,97.56,3.98,arzua


Municipios únicos en el dataset: 90


In [ ]:
# Exportar municipios originales a un txt para normalización manual
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa'

os.makedirs(ruta_txt, exist_ok=True)
archivo_municipios = os.path.join(ruta_txt, 'municipios originales a normalizar.txt')
municipios_originales = sorted(df['municipio'].astype(str).unique())
with open(archivo_municipios, 'w', encoding='utf-8') as f:
    for m in municipios_originales:
        f.write(m + '\n')
print(f"Municipios originales exportados a: {archivo_municipios}")

Municipios originales exportados a: C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa\municipios originales a normalizar.txt


In [8]:
# Mostrar la primera fila como un diccionario columna: valor
primera_fila_dict = df.iloc[0].to_dict()
for k, v in primera_fila_dict.items():
    print(f"{k} = {v}")

fecha = 2000-01-01
municipality_code = 15001
lat = 43.208955250331535
lon = -8.29433479283969
geometry = MULTIPOLYGON (((-8.270063663999963 43.28377445700005, -8.27112978699995 43.27530803900004, -8.256319023999936 43.266584142000056, -8.257109557999968 43.25418760400004, -8.227145668999981 43.24918197000005, -8.241128884999966 43.238214708000044, -8.234642925999935 43.23366026900004, -8.239772432999928 43.22658301200005, -8.233390004999933 43.22235677700007, -8.236784169999964 43.21615154700004, -8.225575037999931 43.216197392000026, -8.225087491999943 43.199718371000074, -8.230994295999949 43.19857454300006, -8.24824522199998 43.20872903000003, -8.25565650699997 43.206470368000055, -8.262913074999972 43.20199476200003, -8.258432316999972 43.18754565000006, -8.276527984999973 43.18083163600005, -8.301341708999928 43.15518605700004, -8.312802575999967 43.15788469600005, -8.318885449999925 43.14610544000004, -8.33582163099993 43.14111058700007, -8.346516126999973 43.13758513200003, -8.3

In [9]:
# Eliminar la columna 'geometry' (no lanza error si no existe)
df.drop(columns=['geometry'], inplace=True, errors='ignore')

### 2.1 Normalizar los nombres de las columnas

In [10]:
# Mostrar nombres originales de columnas
print('Nombres originales de columnas:')
print(list(df.columns))

# Normalizar nombres de columnas a español, minúsculas y descriptivos según los nombres actuales
columnas_renombrar = {
    'municipality_code': 'codigo_municipio',
    'lat': 'latitud',
    'lon': 'longitud',
    'ALLSKY_SFC_SW_DWN': 'radiacion_solar',
    'T2MWET': 'temperatura_humedad',
    'PS': 'presion_superficie',
    'QV2M': 'humedad_especifica',
}

print('\nMapeo de nombres de columnas:')
for k, v in columnas_renombrar.items():
    print(f'{k} -> {v}')

# Renombrar columnas
df.rename(columns=columnas_renombrar, inplace=True)
df.columns = [col.lower() for col in df.columns]

# Eliminar columna de índice si existe
if 'unnamed: 0' in df.columns:
    df.drop(columns=['unnamed: 0'], inplace=True)

print('\nNombres de columnas tras la normalización:')
print(list(df.columns))

Nombres originales de columnas:
['fecha', 'municipality_code', 'lat', 'lon', 'ALLSKY_SFC_SW_DWN', 'T2MWET', 'PS', 'QV2M', 'municipio']

Mapeo de nombres de columnas:
municipality_code -> codigo_municipio
lat -> latitud
lon -> longitud
ALLSKY_SFC_SW_DWN -> radiacion_solar
T2MWET -> temperatura_humedad
PS -> presion_superficie
QV2M -> humedad_especifica

Nombres de columnas tras la normalización:
['fecha', 'codigo_municipio', 'latitud', 'longitud', 'radiacion_solar', 'temperatura_humedad', 'presion_superficie', 'humedad_especifica', 'municipio']


### 3. Visualización y exploración inicial

In [11]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 755178 entries, 0 to 755177
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   fecha                755178 non-null  object 
 1   codigo_municipio     755178 non-null  int64  
 2   latitud              755178 non-null  float64
 3   longitud             755178 non-null  float64
 4   radiacion_solar      755178 non-null  float64
 5   temperatura_humedad  755178 non-null  float64
 6   presion_superficie   755178 non-null  float64
 7   humedad_especifica   755178 non-null  float64
 8   municipio            755178 non-null  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 51.9+ MB


### 4. Cargar el dataset limpio de municipios de Galicia

In [13]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.csv'

# Cargar el archivo Excel de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


### 5. Normalización automática de municipios

In [14]:
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa'
os.makedirs(ruta_txt, exist_ok=True)
# Definir las columnas a usar
col_municipio = 'municipio'  # columna a normalizar en el dataset principal
col_ref = 'municipio'        # columna de referencia en el dataset de municipios
df_ref = df_municipios       # referencia oficial

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# 1. Obtener todos los valores únicos del dataset a normalizar y de la referencia
df[col_municipio] = df[col_municipio].astype(str)
municipios_unicos = set(x for x in df[col_municipio].unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

# 2. Crear mapeo: municipio original -> municipio normalizado (o sugerido, o pendiente)
mapeo = {}
pendientes = []
for m in municipios_unicos:
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        mapeo[m] = ref_norm[clave]
        continue
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[m] = ref_norm[sugerencias[0]]
        continue
    # Probar a invertir el orden de las palabras si hay exactamente dos
    partes = clave.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[m] = ref_norm[invertido]
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[m] = ref_norm[sugerencias_inv[0]]
            continue
    # Si no se encuentra nada, dejar el original y marcar como pendiente
    mapeo[m] = m
    pendientes.append(m)

# 3. Aplicar el mapeo a todo el dataset
df['Municipio_normalizado'] = df[col_municipio].map(mapeo)

# 4. Diagnóstico de diferencias entre dataset y referencia
municipios_normalizados = set(df['Municipio_normalizado'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"Municipios únicos en el dataset de referencia: {len(municipios_referencia)}")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

# 5. Exportar el diccionario de correspondencias y los pendientes
import csv
archivo_diccionario = os.path.join(ruta_txt, 'diccionario_normalizacion_final.txt')
with open(archivo_diccionario, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['original', 'normalizado'])
    for k, v in sorted(mapeo.items()):
        writer.writerow([k, v])

archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in pendientes:
        f.write(f'{m}\n')

print(f"Municipios únicos originales en el dataset principal: {len(municipios_unicos)}")
print(f"Municipios normalizados automáticamente: {len(mapeo) - len(pendientes)}")
print(f"Municipios pendientes de normalizar: {len(pendientes)}")
print(f'Diccionario de normalización exportado a {archivo_diccionario}')
print(f'Listado de pendientes exportado a {archivo_pendientes}')

Municipios únicos en el dataset de referencia: 315
Municipios únicos normalizados en el dataset principal: 90
Municipios de la referencia que NO aparecen en el dataset principal: {'ponteareas', 'vilanova de arousa', 'amoeiro', 'avión', 'o barco de valdeorras', 'sanxenxo', 'boborás', 'moraña', 'mondariz-balneario', 'a cañiza', 'nigrán', 'riós', 'rodeiro', 'as neves', 'forcarei', 'sada', 'baños de molgas', 'crecente', 'pontedeva', 'o saviñao', 'a pontenova', 'portomarín', 'xermade', 'paradela', 'vilalba', 'maceda', 'tomiño', 'lourenzá', 'a rúa', 'negueira de muñiz', 'o grove', 'muíños', 'silleda', 'a arnoia', 'o porriño', 'a teixeira', 'gomesende', 'a pobra do brollón', 'a estrada', 'carballedo', 'lobera', 'manzaneda', 'salvaterra de miño', 'trabada', 'bande', 'verín', 'guitiriz', 'vilardevós', 'xinzo de limia', 'pedrafita do cebreiro', 'mos', 'friol', 'becerreá', 'cervantes', 'sandiás', 'rairiz de veiga', 'covelo', 'o rosal', 'a gudiña', 'vilagarcía de arousa', 'vilar de santos', 'ponte

In [15]:
# Normalización manual de municipios fusionados históricos
df['Municipio_normalizado'] = df['Municipio_normalizado'].replace({'Cesuras': 'oza-cesuras', 'Oza dos Ríos': 'oza-cesuras'})
print("Normalización manual aplicada: 'Cesuras' y 'Oza dos Ríos' ahora son 'oza-cesuras'.")

# Recalcular el conteo tras la corrección manual
municipios_normalizados = set(df['Municipio_normalizado'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
print(f"Municipios únicos normalizados tras corrección manual: {len(municipios_normalizados)}")
print(f"Municipios de la referencia que NO aparecen tras corrección manual: {faltan_en_dataset}")
print(f"Quedan por normalizar: {len(faltan_en_dataset)}")

Normalización manual aplicada: 'Cesuras' y 'Oza dos Ríos' ahora son 'oza-cesuras'.
Municipios únicos normalizados tras corrección manual: 90
Municipios de la referencia que NO aparecen tras corrección manual: {'ponteareas', 'vilanova de arousa', 'amoeiro', 'avión', 'o barco de valdeorras', 'sanxenxo', 'boborás', 'moraña', 'mondariz-balneario', 'a cañiza', 'nigrán', 'riós', 'rodeiro', 'as neves', 'forcarei', 'sada', 'baños de molgas', 'crecente', 'pontedeva', 'o saviñao', 'a pontenova', 'portomarín', 'xermade', 'paradela', 'vilalba', 'maceda', 'tomiño', 'lourenzá', 'a rúa', 'negueira de muñiz', 'o grove', 'muíños', 'silleda', 'a arnoia', 'o porriño', 'a teixeira', 'gomesende', 'a pobra do brollón', 'a estrada', 'carballedo', 'lobera', 'manzaneda', 'salvaterra de miño', 'trabada', 'bande', 'verín', 'guitiriz', 'vilardevós', 'xinzo de limia', 'pedrafita do cebreiro', 'mos', 'friol', 'becerreá', 'cervantes', 'sandiás', 'rairiz de veiga', 'covelo', 'o rosal', 'a gudiña', 'vilagarcía de arou

### 6. Exportar el dataset final con municipios normalizados

In [ ]:
# Sustituir la columna 'municipio' por los valores normalizados y eliminar la auxiliar
df['municipio'] = df['Municipio_normalizado']
if 'Municipio_normalizado' in df.columns:
    df.drop(columns=['Municipio_normalizado'], inplace=True)
print("Columna 'municipio' actualizada con los valores normalizados y columna auxiliar eliminada.")

Columna 'municipio' actualizada con los valores normalizados y columna auxiliar eliminada.


In [ ]:
# Exportar el dataframe final con municipios normalizados (sin columnas duplicadas)

ruta_export = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, 'nasa galicia municipios normalizados.csv')

# Eliminar columnas duplicadas si las hubiera
df = df.loc[:, ~df.columns.duplicated()]

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final exportado como {archivo_export}')

Dataset final exportado como C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa\nasa galicia municipios normalizados.csv


In [ ]:
# Comprobación final: número de municipios únicos en el CSV exportado vs referencia

csv_exportado = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\05 - Nasa\nasa galicia municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)

municipios_exportados = set(df_exportado['municipio'].dropna().unique())
print(f"Municipios únicos en el CSV exportado: {len(municipios_exportados)}")
print(f"Municipios únicos en la referencia oficial: {len(municipios_referencia)}")

if municipios_exportados == municipios_referencia:
    print('¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!')
else:
    diferencia = municipios_exportados.symmetric_difference(municipios_referencia)
    print(f"Diferencias encontradas: {diferencia}")

Municipios únicos en el CSV exportado: 90
Municipios únicos en la referencia oficial: 315
Diferencias encontradas: {'amoeiro', 'boborás', 'moraña', 'mondariz-balneario', 'a cañiza', 'nigrán', 'riós', 'pontedeva', 'o saviñao', 'a pontenova', 'portomarín', 'paradela', 'vilalba', 'negueira de muñiz', 'o grove', 'muíños', 'silleda', 'o porriño', 'a teixeira', 'gomesende', 'a estrada', 'carballedo', 'lobera', 'trabada', 'salvaterra de miño', 'bande', 'guitiriz', 'mos', 'sandiás', 'rairiz de veiga', 'vilar de santos', 'ponte caldelas', 'melón', 'carballeda de valdeorras', 'a guarda', 'toén', 'a mezquita', 'calvos de randín', 'cervo', 'viana do bolo', 'ourol', 'cerdedo-cotobade', 'agolada', 'ribadeo', 'o páramo', 'cesuras', 'o incio', 'castro caldelas', 'a merca', 'beade', 'piñor', 'vilamartín de valdeorras', 'o valadouro', 'ribadumia', 'sober', 'pontevedra', 'vilar de barrio', 'os blancos', 'vilamarín', 'a illa de arousa', 'culleredo', 'allariz', 'vilarmaior', 'san amaro', 'mazaricos', 'rami